# CS-781 Phase 1 — Notebook D: continue training your best model + better decoding

**No new architecture.** Uses YOUR trained model from session 1 and YOUR `models.py`. Evidence from your results:
* validation loss was still falling when every 4.5 h run ended (best checkpoint = last one) → **more training** should help;
* predictions repeat 4-word phrases 3–5× more than real text (4.7% vs 1.6%) and some collapse into loops (`xxxxxxxx`, `****`) → **block repeats** at generation time;
* local validation ranks your models in the same order as Codabench → choose decoding **on validation**.

Two GPUs in parallel:
| GPU | Job | Output |
|---|---|---|
| 0 | **Decoding sweep**: your candidate settings on validation prompts with the original checkpoint, scored with BERTScore/ROUGE-L/BLEU-4; best one then generates the test set | `sweep/submission_<arch>_decoding-<best>.zip` (≈ 15–30 min) |
| 1 | **Continued pretraining** of the same checkpoint for `continue_hours`, then generation with the sweep's best setting | `continued/submission_<arch>_continued_decoding-<best>.zip` (≈ continue_hours + 15 min) |

**KV cache:** generation reuses each layer's stored keys/values instead of re-reading the whole prompt for every new token — same outputs as before (verified), about 10× faster. That time can go into training.

**Kaggle settings:** GPU **T4 ×2** · Internet **On** · secret on · **Add Input**: `phase1-data-prep` (as before) **and** your **session-1 version of `phase1-train`** (it contains `runs/decoder/best.pt`).

**Timing:** the whole run takes ≈ `continue_hours` + 30 min. Keep ≥ 2 h before 11:59 PM for Codabench's queue. Codabench allows 10 submissions per day.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## D3. Configuration — read the comments in the cell: every value is explained with its trade-off. Values marked YOUR CHOICE are yours to set.

In [ ]:
import os
import json
import shutil
import datetime

from huggingface_hub import hf_hub_download


# =====================================================================================================
# D3 — YOUR SETTINGS FOR NOTEBOOK D
# =====================================================================================================

CONFIG = {

    # =================================================================================================
    # BASIC SETTINGS
    # =================================================================================================

    "base_arch": "decoder",
    "seed": 42,


    # =================================================================================================
    # PART 1 — CONTINUED TRAINING
    # =================================================================================================

    # Continue training the best decoder checkpoint for AT MOST 4.5 hours (the length of the original run).
    # The AUTO TIME BUDGET block below lowers this automatically if the run starts late, so that it
    # always finishes by 21:30 IST (≥ 2.5 h before the 23:59 deadline, for Codabench's queue).
    #   start 16:30 or earlier -> 4.5 h   |   start 17:30 -> 3.5 h   |   start 18:30 -> 2.5 h
    #
    # The checkpoint from the original run is used as the starting point.
    # A new checkpoint only replaces the previous best if validation loss improves.

    "continue_hours": 4.5,   # maximum; may be lowered below


    "train": {

        # Keep the same effective batch configuration as the original run.
        "block_size": 1024,
        "batch_size": 8,
        "grad_accum": 2,

        # Original learning rate.
        "lr": 3e-4,

        # Decay all the way toward zero by the end of continued training.
        "min_lr": 0.0,

        # Short warm-up after restarting training.
        "warmup_steps": 500,

        # Original AdamW settings.
        "weight_decay": 0.1,
        "betas": [0.9, 0.999],

        # Gradient clipping.
        "grad_clip": 1.0,

        # Logging / validation frequency.
        "log_every": 50,
        "eval_every": 500,
    },


    # =================================================================================================
    # PART 2 — DECODING SWEEP
    # =================================================================================================

    # Use every validation prompt.
    #
    # There are 287 prompts, and the instructions say this gives the
    # most reliable estimate of which decoding configuration is best.

    "sweep_every": 1,


    # These values are fixed because the task requires 128 words.
    "gen_fixed": {
        "target_words": 128,
        "max_new": 320,
        "batch_size": 16,
    },


    # =================================================================================================
    # DECODING CANDIDATES
    # =================================================================================================
    #
    # "current" = your existing decoding configuration.
    #
    # Each additional candidate changes ONE main decoding choice from
    # the current/reference configuration.
    #
    # The sweep will select the candidate with the highest BERTScore.
    #

    "sweep": {

        # ------------------------------------------------------------
        # BASELINE
        # ------------------------------------------------------------

        "current": {
            "temperature": 0.7,
            "top_k": 0,
            "top_p": 0.9,
            "rep_penalty": 1.1,
            "no_repeat_ngram": 0,
        },


        # ------------------------------------------------------------
        # LOWER TEMPERATURE
        # ------------------------------------------------------------
        # More deterministic.
        # Expected: better exact phrase overlap, potentially more loops.

        "temp_0.5": {
            "temperature": 0.5,
            "top_k": 0,
            "top_p": 0.9,
            "rep_penalty": 1.1,
            "no_repeat_ngram": 0,
        },


        # ------------------------------------------------------------
        # GREEDY + REPETITION BLOCKING
        # ------------------------------------------------------------
        # Temperature 0 is deterministic.
        # 3-gram blocking prevents exact repeated 3-token sequences.

        "greedy_ngram3": {
            "temperature": 0.0,
            "top_k": 0,
            "top_p": 1.0,
            "rep_penalty": 1.0,
            "no_repeat_ngram": 3,
        },


        # ------------------------------------------------------------
        # TOP-P 0.95
        # ------------------------------------------------------------
        # Slightly more permissive nucleus sampling.

        "topp_0.95": {
            "temperature": 0.7,
            "top_k": 0,
            "top_p": 0.95,
            "rep_penalty": 1.1,
            "no_repeat_ngram": 0,
        },


        # ------------------------------------------------------------
        # TOP-K 50
        # ------------------------------------------------------------
        # Restricts sampling to the 50 most likely tokens.

        "topk_50": {
            "temperature": 0.7,
            "top_k": 50,
            "top_p": 0.9,
            "rep_penalty": 1.1,
            "no_repeat_ngram": 0,
        },


        # ------------------------------------------------------------
        # STRONGER REPETITION PENALTY
        # ------------------------------------------------------------
        # More aggressive than the current 1.1.

        "rep_1.2": {
            "temperature": 0.7,
            "top_k": 0,
            "top_p": 0.9,
            "rep_penalty": 1.2,
            "no_repeat_ngram": 0,
        },


        # ------------------------------------------------------------
        # 3-GRAM REPETITION BLOCKING
        # ------------------------------------------------------------
        # Keep sampling behaviour unchanged but explicitly prevent
        # repeated 3-token sequences.

        "ngram3": {
            "temperature": 0.7,
            "top_k": 0,
            "top_p": 0.9,
            "rep_penalty": 1.1,
            "no_repeat_ngram": 3,
        },
    },
}


# =====================================================================================================
# AUTO TIME BUDGET — shortens continue_hours if the run starts late
# =====================================================================================================

FINISH_BY_IST = (21, 30)   # the whole run must be done by 21:30 IST
OVERHEAD_H = 0.5           # setup + generating 287 + 1000 continuations + evaluation, after training

_now_ist = datetime.datetime.utcnow() + datetime.timedelta(hours=5, minutes=30)   # Kaggle runs on UTC
_finish_by = _now_ist.replace(hour=FINISH_BY_IST[0], minute=FINISH_BY_IST[1], second=0, microsecond=0)
_available_h = (_finish_by - _now_ist).total_seconds() / 3600 - OVERHEAD_H
_max_h = CONFIG["continue_hours"]
CONFIG["continue_hours"] = round(max(0.5, min(_max_h, _available_h)), 2)

print(f"continue_hours: maximum {_max_h} h | time available {max(_available_h, 0):.2f} h "
      f"-> using {CONFIG['continue_hours']} h")
if _available_h < 1.0:
    print("WARNING: very little time before 21:30 IST. Consider stopping this run after the sweep "
          "and submitting only sweep/submission_*.zip.")


# =====================================================================================================
# WRITE CONFIGURATION
# =====================================================================================================

config_path = "/kaggle/working/run_config_d.json"

with open(config_path, "w") as f:
    json.dump(
        CONFIG,
        f,
        indent=1
    )


# =====================================================================================================
# DOWNLOAD TEST SET
# =====================================================================================================

test_src = hf_hub_download(
    "Exploration-Lab/CS-781-Capstone",
    "Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

shutil.copy(
    test_src,
    "/kaggle/working/test.json"
)


# =====================================================================================================
# TIMING CHECK
# =====================================================================================================

# Kaggle clocks run on UTC.
# Convert to IST manually as required by the notebook.

now_ist = (
    datetime.datetime.utcnow()
    + datetime.timedelta(hours=5, minutes=30)
)

finish_ist = (
    now_ist
    + datetime.timedelta(
        hours=CONFIG["continue_hours"] + 0.5
    )
)

deadline_ist = now_ist.replace(
    hour=23,
    minute=59,
    second=0,
    microsecond=0
)

buffer_h = (
    deadline_ist - finish_ist
).total_seconds() / 3600


# =====================================================================================================
# PRINT CONFIGURATION SUMMARY
# =====================================================================================================

num_prompts = len(
    range(
        0,
        287,
        CONFIG["sweep_every"]
    )
)

print(
    f"Base architecture: {CONFIG['base_arch']}"
)

print(
    f"Continue training: {CONFIG['continue_hours']} hours"
)

print(
    f"Sweep candidates: {list(CONFIG['sweep'].keys())}"
)

print(
    f"Sweep prompts: {num_prompts}"
)

print(
    f"Now (IST): {now_ist:%H:%M}"
)

print(
    f"Expected finish (IST): {finish_ist:%H:%M}"
)

print(
    f"Buffer before 23:59 IST: {buffer_h:.1f} h "
    f"{'OK' if buffer_h >= 2 else 'TOO TIGHT: lower continue_hours'}"
)

print(
    f"\nConfig saved to: {config_path}"
)

## D4. Pipeline code (phase1-train's `lm_common.py` + KV-cache generation + no-repeat-ngram blocking + metric helpers)

In [ ]:
%%writefile lm_common_d.py
"""Same pipeline as phase1-train's lm_common.py, plus: no-repeat-ngram blocking in generation and
metric helpers (same ROUGE-L / BLEU-4 as the official evaluator; BERTScore via the same library)."""
import glob, json, math, os, random, sys, zipfile
from collections import Counter
import numpy as np, torch, torch.nn.functional as F

def find_input(name):
    for root in ("/kaggle/working", "/kaggle/input"):
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found — are the data-prep and session-1 outputs added as input?")

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

WORD_MARK = "".join(chr(b) for b in "\u2581".encode("utf-8"))

class Tok:
    def __init__(self, path=None):
        self.bpe = bpe_tok.load_bpe(path or find_input("bpe.json"))
        self.enc = bpe_tok.FastEncoder(self.bpe)
        self.itos, self.V = self.bpe.itos, len(self.bpe.itos)
        self.pad, self.unk, self.bos, self.eos = (self.bpe.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
        self.word_start = torch.tensor([t.startswith(WORD_MARK) for t in self.itos])
    def encode(self, text):
        return self.enc.encode(text)
    def decode(self, ids):
        return bpe_tok.detokenize([self.itos[i] for i in ids if i >= 4])

def get_batch(data, batch_size, block, device, rng=np.random):
    ix = rng.randint(0, len(data) - block - 1, batch_size)
    x = torch.from_numpy(np.stack([data[i:i + block + 1] for i in ix]).astype(np.int64))
    if str(device).startswith("cuda"):
        x = x.pin_memory()
    return x.to(device, non_blocking=True)

def compute_loss(model, seq):
    """Causal next-token loss (the continued model is a left-to-right model)."""
    logits = model(seq[:, :-1])
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), seq[:, 1:].reshape(-1))

@torch.no_grad()
def eval_loss(model, data, batch_size, block, device, n_batches=20):
    model.eval()
    rng, losses = np.random.RandomState(1234), []
    for _ in range(n_batches):
        seq = get_batch(data, batch_size, block, device, rng)
        with torch.autocast("cuda", dtype=torch.float16):
            losses.append(compute_loss(model, seq).item())
    model.train()
    return float(np.mean(losses))

def sample_next(logits, generated, temperature, top_k, top_p, rep_penalty, banned):
    logits = logits.float().clone()
    logits[:, banned] = -float("inf")
    if rep_penalty != 1.0 and generated.size(1) > 0:
        prev = torch.gather(logits, 1, generated)
        logits.scatter_(1, generated, torch.where(prev > 0, prev / rep_penalty, prev * rep_penalty))
    if temperature <= 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        sorted_logits = sorted_logits.masked_fill(probs.cumsum(-1) - probs > top_p, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)

class KVCacheRunner:
    """Fast incremental forward pass for models built like your DecoderLM (token + learned position
    embeddings -> nn.TransformerEncoderLayer(norm_first=True) stack with a causal mask -> LayerNorm -> head).
    It uses the model's OWN weights and the same arithmetic as model(idx), but keeps every layer's keys and
    values ("KV cache"), so each new token attends to the stored ones instead of re-running the whole
    sequence. Same logits, about 10x faster generation. Other model types fall back to the plain loop."""

    @staticmethod
    def supports(model):
        layers = getattr(model, "layers", None)
        return (getattr(model, "kind", "") == "causal"
                and all(hasattr(model, a) for a in ("tok_emb", "pos_emb", "norm", "head"))
                and layers is not None and len(layers) > 0
                and all(isinstance(l, torch.nn.TransformerEncoderLayer) and l.norm_first for l in layers))

    def __init__(self, model):
        self.m, self.k, self.v, self.t = model, [], [], 0

    def _attention(self, layer, h, i, prefill):
        sa = layer.self_attn
        B, T, D = h.shape
        H = sa.num_heads
        q, k, v = F.linear(h, sa.in_proj_weight, sa.in_proj_bias).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, H, D // H).transpose(1, 2) for t in (q, k, v))
        if prefill:
            self.k.append(k); self.v.append(v)
        else:
            self.k[i] = torch.cat([self.k[i], k], 2); self.v[i] = torch.cat([self.v[i], v], 2)
        o = F.scaled_dot_product_attention(q, self.k[i], self.v[i], is_causal=prefill)
        return sa.out_proj(o.transpose(1, 2).reshape(B, T, D))

    def _run(self, idx, start, prefill):
        m = self.m
        x = m.tok_emb(idx) + m.pos_emb(torch.arange(start, start + idx.size(1), device=idx.device))
        for i, layer in enumerate(m.layers):
            x = x + self._attention(layer, layer.norm1(x), i, prefill)
            x = x + layer.linear2(layer.activation(layer.linear1(layer.norm2(x))))
        return m.head(m.norm(x[:, -1]))

    def prefill(self, idx):
        self.k, self.v, self.t = [], [], idx.size(1)
        return self._run(idx, 0, True)

    def step(self, last_token):
        out = self._run(last_token, self.t, False)
        self.t += 1
        return out

@torch.no_grad()
def generate(model, tok, prompts, device, temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.0,
             no_repeat_ngram=0, target_words=128, use_cache=True, max_new=320, batch_size=16, seed=0, log_every=10):
    """Left-to-right generation (as in phase1-train) + optional no-repeat-ngram blocking: with
    no_repeat_ngram = n > 0, a token is forbidden if it would complete an n-token sequence that the
    continuation has already produced (the prompt itself is not counted, legal phrases may recur there)."""
    torch.manual_seed(seed)
    model.eval()
    ids = [tok.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    ws = tok.word_start.to(device)
    banned = torch.tensor([tok.pad, tok.unk, tok.bos, tok.eos], device=device)
    max_prompt = model.max_len - max_new
    n = int(no_repeat_ngram or 0)
    cached = use_cache and KVCacheRunner.supports(model)
    out = [""] * len(ids)
    for bi, b0 in enumerate(range(0, len(order), batch_size)):
        bidx = order[b0:b0 + batch_size]
        L = min(min(len(ids[i]) for i in bidx), max_prompt)
        prompt = torch.tensor([ids[i][-L:] for i in bidx], device=device)
        B = len(bidx)
        gen = torch.empty(B, 0, dtype=torch.long, device=device)
        seen = [dict() for _ in range(B)]          # per row: (n-1 previous tokens) -> set of next tokens used
        hist = [[] for _ in range(B)]
        runner = KVCacheRunner(model) if cached else None
        for step in range(max_new):
            with torch.autocast("cuda", dtype=torch.float16):
                if cached:
                    logits = (runner.prefill(prompt) if step == 0 else runner.step(gen[:, -1:])).float()
                else:
                    logits = model(torch.cat([prompt, gen], 1))[:, -1].float()
            if n > 0 and gen.size(1) >= n - 1:
                for j in range(B):
                    nxt_banned = seen[j].get(tuple(hist[j][-(n - 1):]) if n > 1 else ())
                    if nxt_banned:
                        logits[j, list(nxt_banned)] = -float("inf")
            nxt = sample_next(logits, gen, temperature, top_k, top_p, rep_penalty, banned)
            gen = torch.cat([gen, nxt[:, None]], 1)
            if n > 0:
                for j, t in enumerate(nxt.tolist()):
                    hist[j].append(t)
                    if len(hist[j]) >= n:
                        seen[j].setdefault(tuple(hist[j][-n:-1]), set()).add(t)
            if (ws[gen].sum(1) > target_words).all():
                break
        for j, i in enumerate(bidx):
            words = tok.decode(gen[j].tolist()).split()[:target_words]
            out[i] = " ".join(words) if words else "the"
        if log_every and bi % log_every == 0:
            print(f"  generated {min(b0 + batch_size, len(order))}/{len(order)}", flush=True)
    return out

def write_submission(preds, ids, json_path, zip_path=None):
    missing = [i for i in ids if i not in preds]
    assert not missing, f"{len(missing)} ids missing"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({i: {"text": preds[i]} for i in ids}, f, ensure_ascii=False, indent=2)
    if zip_path:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(json_path, arcname="prediction.json")

# ---------------- metrics: same definitions as the official evaluate_phase1.py ----------------
def _lcs(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")

def rouge_l(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt:
        return 0.0
    l = _lcs(pt, rt)
    if not l:
        return 0.0
    pp, rr = l / len(pt), l / len(rt)
    return 2 * pp * rr / (pp + rr)

def bleu4(preds, refs):
    def ng(t, n):
        return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1)) if len(t) >= n else Counter()
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        p, r = p.split(), r.split(); pl += len(p); rl += len(r)
        for n in range(1, 5):
            pn, rn = ng(p, n), ng(r, n); totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    bp = 1.0 if pl > rl else math.exp(1.0 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / 4)

def repeat_rate(text, n=4):
    w = text.split(); g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 1 - len(set(g)) / len(g) if g else 0.0

## D5. YOUR `models.py` (unchanged copy of phase1-train cell B5 — your own code)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

## D6. Check: checkpoint loads, candidates are valid (CPU only, a few seconds)

In [ ]:
# Quick check before launching: both checkpoints load, the no-repeat-ngram blocking works, nothing is left on GPU.
import json, importlib, torch
import lm_common_d as L, models
importlib.reload(L); importlib.reload(models)
C = json.load(open("/kaggle/working/run_config_d.json"))
tok = L.Tok()
path = L.find_input(f"{C['base_arch']}/best.pt")
ck = torch.load(path, map_location="cpu")
m = models.build_model(C["base_arch"], vocab_size=tok.V, **ck["cfg"]); m.load_state_dict(ck["model"])
print(f"checkpoint: {path}\n  step {ck['step']} | val_loss {ck['val_loss']:.3f} | "
      f"{sum(p.numel() for p in m.parameters())/1e6:.1f}M params | cfg {ck['cfg']}")
for name, s in C["sweep"].items():
    unknown = set(s) - {"temperature", "top_k", "top_p", "rep_penalty", "no_repeat_ngram"}
    assert not unknown, f"candidate {name}: unknown keys {unknown}"
print("candidates OK:", list(C["sweep"]))
del m; torch.cuda.empty_cache()

## D7. Scripts: decoding sweep (GPU 0) and continued pretraining (GPU 1)

In [ ]:
%%writefile sweep_d.py
"""GPU 0: try every candidate decoding setting on a validation subset with the ORIGINAL checkpoint,
score each with BERTScore / ROUGE-L / BLEU-4, pick the best by BERTScore, then write a test submission."""
import argparse, json, os, time
import numpy as np, torch
import lm_common_d as L
import models

ap = argparse.ArgumentParser(); ap.add_argument("--gpu", type=int, default=0); args = ap.parse_args()
C = json.load(open("/kaggle/working/run_config_d.json"))
arch, dev = C["base_arch"], f"cuda:{args.gpu}"
torch.cuda.set_device(dev)
OUT = "/kaggle/working/sweep"; os.makedirs(OUT, exist_ok=True)

tok = L.Tok()
ck = torch.load(L.find_input(f"{arch}/best.pt"), map_location=dev)
model = models.build_model(arch, vocab_size=tok.V, **ck["cfg"]).to(dev).eval()
model.load_state_dict(ck["model"])
print(f"loaded {arch} checkpoint (step {ck.get('step')}, val_loss {ck.get('val_loss'):.3f})", flush=True)

vp = json.load(open(L.find_input("val_prompts.json"))); vt = json.load(open(L.find_input("val_truth.json")))
ids = list(vp)[::C["sweep_every"]]
prompts, refs = [vp[i]["text"] for i in ids], [vt[i]["text"] for i in ids]

from bert_score import BERTScorer          # same defaults as the official evaluator: roberta-large, no idf, no rescale
scorer = BERTScorer(lang="en", device=dev)

results = {}
for name, s in C["sweep"].items():
    t0 = time.time()
    preds = L.generate(model, tok, prompts, dev, **{**C["gen_fixed"], **s}, log_every=0)
    _, _, f1 = scorer.score(preds, refs, batch_size=16)
    results[name] = {"setting": s, "bertscore_f1": round(float(f1.mean()), 5),
                     "rouge_l": round(float(np.mean([L.rouge_l(p, r) for p, r in zip(preds, refs)])), 5),
                     "bleu4": round(L.bleu4(preds, refs), 5),
                     "repeat_4gram_pct": round(100 * float(np.mean([L.repeat_rate(p) for p in preds])), 2),
                     "min_words": min(len(p.split()) for p in preds), "seconds": round(time.time() - t0)}
    print(name, results[name], flush=True)
    json.dump({"results": results}, open(f"{OUT}/sweep_partial.json", "w"), indent=1)

best = max(results, key=lambda k: results[k]["bertscore_f1"])
json.dump({"results": results, "best": best, "best_setting": C["sweep"][best], "n_prompts": len(ids)},
          open(f"{OUT}/sweep_results.json", "w"), indent=1)
print(f"BEST by validation BERTScore: {best} -> {C['sweep'][best]}", flush=True)
del scorer; torch.cuda.empty_cache()

tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
preds = L.generate(model, tok, [tp[i]["text"] for i in tids], dev, **{**C["gen_fixed"], **C["sweep"][best]})
L.write_submission(dict(zip(tids, preds)), tids, f"{OUT}/prediction.json", f"{OUT}/submission_{arch}_decoding-{best}.zip")
print("sweep done", flush=True)

In [ ]:
%%writefile continue_d.py
"""GPU 1: continue pretraining the chosen checkpoint for continue_hours (learning rate re-warmed, then
cosine-decayed), keep the best checkpoint by validation loss, then generate with the sweep's best setting."""
import argparse, json, math, os, random, time
import numpy as np, torch
import lm_common_d as L
import models

ap = argparse.ArgumentParser(); ap.add_argument("--gpu", type=int, default=1); args = ap.parse_args()
C = json.load(open("/kaggle/working/run_config_d.json"))
arch, T_, dev = C["base_arch"], C["train"], f"cuda:{args.gpu}"
torch.cuda.set_device(dev)
random.seed(C["seed"] + 1); np.random.seed(C["seed"] + 1); torch.manual_seed(C["seed"] + 1)
OUT = "/kaggle/working/continued"; os.makedirs(OUT, exist_ok=True)
t_start = time.time()
log = open(f"{OUT}/log.jsonl", "a")
def say(**kw):
    kw["t"] = round(time.time() - t_start); log.write(json.dumps(kw) + "\n"); log.flush()
    print("continue", kw, flush=True)

tok = L.Tok()
train_data = np.memmap(L.find_input("train.bin"), dtype=np.uint16, mode="r")
val_data = np.memmap(L.find_input("val.bin"), dtype=np.uint16, mode="r")
ck = torch.load(L.find_input(f"{arch}/best.pt"), map_location=dev)
model = models.build_model(arch, vocab_size=tok.V, **ck["cfg"]).to(dev)
model.load_state_dict(ck["model"])
block = T_["block_size"]

decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": T_["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}], lr=T_["lr"], betas=tuple(T_["betas"]))
scaler = torch.amp.GradScaler("cuda")
budget = C["continue_hours"] * 3600

def lr_at(step, frac):
    if step < T_["warmup_steps"]:
        return T_["lr"] * (step + 1) / T_["warmup_steps"]
    return T_["min_lr"] + 0.5 * (T_["lr"] - T_["min_lr"]) * (1 + math.cos(math.pi * min(frac, 1.0)))

def save(path, step, vl):
    torch.save({"model": model.state_dict(), "arch": arch, "cfg": ck["cfg"], "step": step, "val_loss": vl,
                "continued_from_step": ck.get("step")}, path)

best = L.eval_loss(model, val_data, T_["batch_size"], block, dev)
say(event="eval", step=0, val_loss=round(best, 4), tokens=0, note="starting checkpoint")
save(f"{OUT}/best_continued.pt", 0, best)
step, tokens = 0, 0
model.train()
while (frac := (time.time() - t_start) / budget) < 1:
    lr = lr_at(step, frac)
    for g in opt.param_groups:
        g["lr"] = lr
    tot = 0.0
    for _ in range(T_["grad_accum"]):
        seq = L.get_batch(train_data, T_["batch_size"], block, dev)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = L.compute_loss(model, seq) / T_["grad_accum"]
        scaler.scale(loss).backward(); tot += loss.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), T_["grad_clip"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    step += 1; tokens += T_["batch_size"] * T_["grad_accum"] * block
    if step % T_["log_every"] == 0:
        say(event="train", step=step, loss=round(tot, 4), lr=lr, tokens=tokens)
    if step % T_["eval_every"] == 0:
        vl = L.eval_loss(model, val_data, T_["batch_size"], block, dev)
        say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
        if vl < best:
            best = vl; save(f"{OUT}/best_continued.pt", step, vl)
vl = L.eval_loss(model, val_data, T_["batch_size"], block, dev)
say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
if vl < best:
    best = vl; save(f"{OUT}/best_continued.pt", step, vl)

# ---- generation with the decoding setting chosen by the sweep (waits for it if still running) ----
del opt; torch.cuda.empty_cache()
model.load_state_dict(torch.load(f"{OUT}/best_continued.pt", map_location=dev)["model"])
sweep_file = "/kaggle/working/sweep/sweep_results.json"
for _ in range(240):                       # wait up to 2 h
    if os.path.exists(sweep_file):
        break
    time.sleep(30)
if os.path.exists(sweep_file):
    chosen = json.load(open(sweep_file)); name, setting = chosen["best"], chosen["best_setting"]
else:
    name, setting = "current", C["sweep"]["current"]
say(event="decoding", chosen=name, setting=setting)
gen = {**C["gen_fixed"], **setting}
vp = json.load(open(L.find_input("val_prompts.json"))); vids = list(vp)
preds = L.generate(model, tok, [vp[i]["text"] for i in vids], dev, **gen)
L.write_submission(dict(zip(vids, preds)), vids, f"{OUT}/val_prediction.json")
tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
preds = L.generate(model, tok, [tp[i]["text"] for i in tids], dev, **gen)
L.write_submission(dict(zip(tids, preds)), tids, f"{OUT}/prediction.json",
                   f"{OUT}/submission_{arch}_continued_decoding-{name}.zip")
say(event="done", best_val_loss=best, steps=step, tokens=tokens)

## D8. Launch both in parallel; progress every 2 min

In [ ]:
import subprocess, sys, time, torch
assert torch.cuda.device_count() >= 2, "needs GPU T4 x2 (sweep on GPU 0, continued training on GPU 1)"
!pip install -q bert-score
procs = []
for script, gpu in [("sweep_d.py", 0), ("continue_d.py", 1)]:
    out = open(f"/kaggle/working/{script}.out", "w")
    procs.append((script, subprocess.Popen([sys.executable, script, "--gpu", str(gpu)], stdout=out,
                                           stderr=subprocess.STDOUT, cwd="/kaggle/working")))
while any(p.poll() is None for _, p in procs):
    time.sleep(120)
    for script, _ in procs:
        lines = open(f"/kaggle/working/{script}.out").read().strip().splitlines()
        print(time.strftime("%H:%M"), "|", script, "|", lines[-1][:240] if lines else "(no output yet)", flush=True)
for script, p in procs:
    print(f"{script}: exit code {p.returncode}")
    if p.returncode != 0:
        print(open(f"/kaggle/working/{script}.out").read()[-3000:])

## D9. Results: decoding sweep table, continued-training curves, official validation score, zip check

In [ ]:
import json
r = json.load(open("/kaggle/working/sweep/sweep_results.json"))
print(f"Decoding sweep on {r['n_prompts']} validation prompts (ORIGINAL checkpoint):\n")
print(f"{'candidate':16s} {'BERTScore':>9s} {'ROUGE-L':>8s} {'BLEU-4':>7s} {'rep4%':>6s} {'minW':>5s}  setting")
for k, v in sorted(r["results"].items(), key=lambda kv: -kv[1]["bertscore_f1"]):
    print(f"{k:16s} {v['bertscore_f1']:9.4f} {v['rouge_l']:8.4f} {v['bleu4']:7.4f} {v['repeat_4gram_pct']:6.1f} {v['min_words']:5d}  {v['setting']}")
print("\nchosen:", r["best"], "(real text has about 1.6% repeated 4-grams)")

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open("/kaggle/working/continued/log.jsonl")]
ev = [r for r in rows if r["event"] == "eval"]; tr = [r for r in rows if r["event"] == "train"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot([r["tokens"] / 1e6 for r in tr], [r["loss"] for r in tr]); ax[0].set(title="Continued training loss", xlabel="extra tokens (M)")
ax[1].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-"); ax[1].set(title="Validation loss (step 0 = original checkpoint)", xlabel="extra tokens (M)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/kaggle/working/continued_curves.png", dpi=200); plt.show()
print("validation loss: start", ev[0]["val_loss"], "-> best", min(r["val_loss"] for r in ev))

In [ ]:
%%writefile evaluate_phase1.py
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)

In [ ]:
import lm_common_d as L
VAL_TRUTH = L.find_input("val_truth.json")
print("===== continued model, full validation set (287), official evaluator")
!python evaluate_phase1.py --truth {VAL_TRUTH} --prediction /kaggle/working/continued/val_prediction.json

In [ ]:
import zipfile, json, glob
test_ids = set(json.load(open("/kaggle/working/test.json")))
for z in sorted(glob.glob("/kaggle/working/*/submission_*.zip")):
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    wc = [len(v["text"].split()) for v in pred.values()]
    ok = names == ["prediction.json"] and set(pred) == test_ids and all(v["text"].strip() for v in pred.values())
    print(f"{z}: {'OK' if ok else 'PROBLEM'} | ids={len(pred)} | words min/mean/max = {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)}")